# Домашнее задание 1. Нейронные сети, кластеризация и графы

## Дополнительные главы машинного обучения

В части A предскажите год выпуска трека по аудиопризнакам Year Prediction MSD и исследуйте
кластеры треков. В части B определите тему статьи в графе цитирований Cora и проверьте,
что модели даёт структура связей.

**Обязательная часть: 10 баллов, по 5 за каждую часть.**
Бонусы: до 2 баллов за эксперименты с MLP и 0,1 за анекдот. Максимум с бонусами: 12,1 балла.

<div style="background-color: crimson; color: white; padding: 16px 20px; border-radius: 8px; font-size: 1.25em; font-weight: bold; line-height: 1.5;">
Ответы на вопросы обязательны.<br>
За код без ответов на вопросы максимальная оценка составляет 1/3 от максимального балла за задание. <br>
Постарайтесь написать комменты сами, подумайте пожалуйста. Не надо нейрослопить
</div>

Величина score не влияет на балл. Оцениваются корректность реализации, постановка экспериментов
и обоснование выводов.

In [ ]:
# %pip install numpy pandas scipy scikit-learn matplotlib torch networkx "node2vec==0.5.0" "gensim>=4.4"

%matplotlib inline

In [ ]:
from pathlib import Path
from zipfile import ZipFile
from urllib.request import urlretrieve
import copy
import random
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

SEED = sum(ord(c) for c in "six_seven") # вы не подумайте, мне не 5 лет, мне 6

N_TRAIN, N_VAL, N_TEST = 15_000, 3_000, 5_000
EPOCHS = 30
BATCH_SIZE = 256
LR = 1e-3
DATA_DIR = Path('data_mlp_homework')
DATA_DIR.mkdir(exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_num_threads(min(4, torch.get_num_threads()))

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
print('PyTorch:', torch.__version__, '| Устройство:', DEVICE)

### Общие функции: запуск, оптимизатор и сохранение

Для каждого запуска сохраняйте параметры и итоговые метрики. Для MLP и GCN дополнительно
сохраняйте метрики каждой эпохи и лучшие веса.
Веса в Git добавлять не нужно, код их сохранения оставьте в ноутбуке.
`make_run_folder` и `make_optimizer` уже готовы; `save_progress` заполняется в разделе 2 один раз
и затем используется в обеих частях.

In [ ]:
RESULTS_DIR = Path('results_homework')
RESULTS_DIR.mkdir(exist_ok=True)
music_runs, cora_runs = {}, {}

def make_run_folder(config):
    number = 1
    while (RESULTS_DIR / f'run_{number:03d}').exists():
        number += 1
    folder = RESULTS_DIR / f'run_{number:03d}'
    folder.mkdir()
    pd.Series(config).to_json(folder / 'config.json', force_ascii=False, indent=2)
    return folder

def make_optimizer(model, name, lr, weight_decay=0.0):
    options = dict(lr=lr, weight_decay=weight_decay)
    if name == 'SGD':
        options['momentum'] = 0.9
    return {'Adam': torch.optim.Adam, 'SGD': torch.optim.SGD}[name](model.parameters(), **options)

# Часть A. Музыкальные данные - 5 баллов

Используем Year Prediction MSD:
90 аудиопризнаков и год выпуска. Фиксированная выборка содержит 15 000 train, 3 000 validation
и 5 000 test. При наличии файла выборки в `data_mlp_homework` исходный архив скачивать не нужно.
Статистики преобразований оценивайте на train. Основные метрики: MSE в годах² и RMSE в годах.

In [ ]:
MSD_URL = 'https://archive.ics.uci.edu/static/public/203/yearpredictionmsd.zip'
MSD_ARCHIVE = DATA_DIR / 'yearpredictionmsd.zip'

def load_msd_sample(archive, data_dir):
    official_train = 463_715
    total_rows = 515_345
    cache = data_dir / f'sample_{SEED}_{N_TRAIN}_{N_VAL}_{N_TEST}.npz'
    if cache.exists():
        with np.load(cache) as saved:
            return {key: saved[key] for key in saved.files}

    if not archive.exists():
        print('Загружаем исходный архив UCI...')
        urlretrieve(MSD_URL, archive)

    rng = np.random.default_rng(SEED)
    development_ids = np.sort(rng.choice(official_train, N_TRAIN + N_VAL, replace=False))
    test_ids = np.sort(rng.choice(np.arange(official_train, total_rows), N_TEST, replace=False))
    wanted = np.sort(np.concatenate([development_ids, test_ids]))
    selected = np.empty((len(wanted), 91), dtype=np.float32)
    offset = 0
    with ZipFile(archive) as zf:
        members = [name for name in zf.namelist() if name.endswith('YearPredictionMSD.txt')]
        with zf.open(members[0]) as source:
            for chunk in pd.read_csv(source, header=None, dtype=np.float32, chunksize=20_000):
                lo = np.searchsorted(wanted, offset)
                hi = np.searchsorted(wanted, offset + len(chunk))
                if hi > lo:
                    selected[lo:hi] = chunk.iloc[wanted[lo:hi] - offset].to_numpy()
                offset += len(chunk)

    train_ids, val_ids = train_test_split(development_ids, test_size=N_VAL, random_state=SEED)
    splits = {'train': train_ids, 'val': val_ids, 'test': test_ids}
    result = {}
    for split, row_ids in splits.items():
        rows = selected[np.searchsorted(wanted, row_ids)]
        result[f'X_{split}'] = rows[:, 1:].copy()
        result[f'y_{split}'] = rows[:, 0].copy()
        result[f'ids_{split}'] = row_ids
    np.savez_compressed(cache, **result)
    return result

sample = load_msd_sample(MSD_ARCHIVE, DATA_DIR)
X_train, y_train = sample['X_train'], sample['y_train']
X_val, y_val = sample['X_val'], sample['y_val']
print('Train:', X_train.shape, '| Validation:', X_val.shape)

## 1. Базовые модели для музыки - 1 балл

Найдите лучший постоянный прогноз для MSE и обучите Ridge с `alpha=1`.
Стандартизируйте признаки, оценив параметры по train. Сохраните `constant_year`, `feature_scaler`,
`X_train_std`, `X_val_std`, `ridge` и validation-таблицу `music_baselines` со столбцами
`name`, `val_mse`, `val_rmse`. Запишите эту таблицу в `RESULTS_DIR / 'music_baselines.csv'`.
Test в этом разделе не используйте.

**Допишите следующий блок.**

In [ ]:
# >>> ВАШ КОД: константа, StandardScaler, Ridge и таблица music_baselines.
# <<< ВАШ КОД

**Ответьте:** что показало сравнение с постоянным прогнозом? Какой validation RMSE получила Ridge?

**Ваш ответ:**

## 2. MLP и общие функции обучения - 1,5 балла

Реализуйте MLP, преобразование года, шаг обучения и сохранение результатов.
Эти функции используются дальше без повторных заданий на тот же код.

- `make_mlp`, 0,5 балла: `Linear → ReLU → [Dropout] → Linear`.
- `scale_year`, 0,25 балла: min-max по `y_train`, `inverse=True` возвращает годы.
- `train_step`, 0,5 балла: перенос батча, обнуление градиентов, forward, loss, backward и шаг оптимизатора.
- `save_progress`, 0,25 балла: CSV после каждой эпохи и веса при улучшении validation-метрики.

В MLP должна поддерживаться произвольная размерность входа и выхода: в части B будет
использована та же функция с другими размерами. Dropout добавляйте после ReLU, если `dropout > 0`.
Последняя активация отсутствует.

**Допишите сборку модели.**

In [ ]:
def make_mlp(in_features=90, out_features=1, hidden=128, dropout=0.0):
    # >>> ВАШ КОД: соберите и верните nn.Sequential.
    pass
    # <<< ВАШ КОД

**Допишите прямое и обратное преобразование года.**

In [ ]:
y_min = float(y_train.min())
y_scale = float(y_train.max() - y_train.min())

def scale_year(values, inverse=False):
    # >>> ВАШ КОД: используйте y_min и y_scale, сохраните поддержку NumPy и torch.
    pass
    # <<< ВАШ КОД

**Допишите один шаг обучения. Этот код вызывается также при обучении на Cora.**
Верните loss как torch-тензор и предсказания модели. Модель уже переведена в режим `train()`.

In [ ]:
def train_step(model, optimizer, criterion, features, target, device):
    # >>> ВАШ КОД: верните loss и предсказания модели.
    pass
    # <<< ВАШ КОД

**Допишите сохранение результатов.** `history` - список словарей с метриками по эпохам.
Записывайте его в `folder / 'history.csv'`. При `is_best=True` сохраняйте `model.state_dict()`
в `folder / 'best_model.pt'`. Функция используется в обеих частях.

In [ ]:
def save_progress(history, model, folder, is_best):
    # >>> ВАШ КОД: сохраните history.csv после каждой эпохи.
    pass
    # <<< ВАШ КОД
    if is_best:
        # >>> ВАШ КОД: сохраните best_model.pt.
        pass
        # <<< ВАШ КОД

### Готовые функции для музыкальной регрессии

Модель выбирается по validation MSE в исходной шкале годов. История и лучшие веса сохраняются
через вашу `save_progress`. На train метрика считается в `eval()`.

In [ ]:
def music_loaders(x_tr, t_tr, x_va, t_va, seed, batch_size):
    train_ds = TensorDataset(torch.as_tensor(x_tr, dtype=torch.float32), torch.as_tensor(t_tr))
    val_ds = TensorDataset(torch.as_tensor(x_va, dtype=torch.float32), torch.as_tensor(t_va))
    generator = torch.Generator().manual_seed(seed)
    return (DataLoader(train_ds, batch_size=batch_size, shuffle=True, generator=generator),
            DataLoader(val_ds, batch_size=batch_size, shuffle=False))

def music_predictions(model, features, decode, batch_size=BATCH_SIZE):
    model.eval()
    predictions = []
    with torch.no_grad():
        for start in range(0, len(features), batch_size):
            x = torch.as_tensor(features[start:start + batch_size], dtype=torch.float32, device=DEVICE)
            predictions.append(decode(model(x)).cpu().numpy().reshape(-1))
    return np.concatenate(predictions)

In [ ]:
def music_validation(model, loader, decode, true_years):
    model.eval()
    total, predictions = 0.0, []
    with torch.no_grad():
        for x, target in loader:
            output = model(x.to(DEVICE))
            total += nn.functional.mse_loss(output, target.to(DEVICE)).item() * len(x)
            predictions.append(decode(output).cpu().numpy().reshape(-1))
    prediction = np.concatenate(predictions)
    mse = mean_squared_error(true_years, prediction)
    return total / len(loader.dataset), float(mse)

def fit_music(model, optimizer, loaders, x_tr, decode, epochs, folder, batch_size):
    history, best_state, best_mse, best_epoch = [], None, np.inf, None
    criterion = nn.MSELoss()
    for epoch in range(1, epochs + 1):
        model.train()
        total = 0.0
        for x, target in loaders[0]:
            loss, _ = train_step(model, optimizer, criterion, x, target, DEVICE)
            total += loss.item() * len(x)
        train_mse = mean_squared_error(y_train, music_predictions(model, x_tr, decode, batch_size))
        val_loss, val_mse = music_validation(model, loaders[1], decode, y_val)
        history.append(dict(epoch=epoch, train_loss=total / len(loaders[0].dataset),
                            val_loss=val_loss, train_mse=train_mse, val_mse=val_mse,
                            train_rmse=np.sqrt(train_mse), val_rmse=np.sqrt(val_mse)))
        is_best = val_mse < best_mse
        if is_best:
            best_mse, best_epoch = val_mse, epoch
            best_state = copy.deepcopy(model.state_dict())
        save_progress(history, model, folder, is_best)
    model.load_state_dict(best_state)
    return best_mse, best_epoch

In [ ]:
def run_music(name, standardized=True, normalized_target=True, hidden=128, dropout=0.0,
              epochs=None, lr=None, batch_size=None, optimizer_name='Adam', seed=SEED,
              sigmoid=False):
    epochs = EPOCHS if epochs is None else epochs
    lr = LR if lr is None else lr
    batch_size = BATCH_SIZE if batch_size is None else batch_size
    seed_everything(seed)
    x_tr = X_train_std if standardized else X_train
    x_va = X_val_std if standardized else X_val
    target = scale_year if normalized_target else (lambda values: values)
    decode = (lambda values: scale_year(values, inverse=True)) if normalized_target else (lambda values: values)
    t_tr = np.asarray(target(y_train), dtype=np.float32).reshape(-1, 1)
    t_va = np.asarray(target(y_val), dtype=np.float32).reshape(-1, 1)
    model = make_mlp(hidden=hidden, dropout=dropout)
    if sigmoid:
        model = nn.Sequential(model, nn.Sigmoid())
    model = model.to(DEVICE)
    optimizer = make_optimizer(model, optimizer_name, lr)
    loaders = music_loaders(x_tr, t_tr, x_va, t_va, seed, batch_size)
    config = dict(part='music', name=name, standardized=standardized, normalized_target=normalized_target,
                  hidden=hidden, dropout=dropout, epochs=epochs, lr=lr, batch_size=batch_size,
                  optimizer=optimizer_name, seed=seed, sigmoid=sigmoid, target_min=y_min, target_range=y_scale)
    folder = make_run_folder(config)
    started = time.perf_counter()
    val_mse, best_epoch = fit_music(model, optimizer, loaders, x_tr, decode, epochs, folder, batch_size)
    record = dict(config, val_mse=val_mse, val_rmse=np.sqrt(val_mse), best_epoch=best_epoch,
                  seconds=time.perf_counter() - started,
                  parameters=sum(p.numel() for p in model.parameters()), artifacts=str(folder))
    music_runs[name] = dict(model=model.cpu(), decode=decode, record=record, config=config)
    pd.Series(record).to_json(folder / 'result.json', force_ascii=False, indent=2)
    print(name, '| validation RMSE:', round(record['val_rmse'], 3))
    return record

def music_table():
    table = pd.DataFrame([r['record'] for r in music_runs.values()]).set_index('name')
    return table[['val_mse', 'val_rmse', 'best_epoch', 'epochs', 'hidden', 'dropout',
                  'optimizer', 'lr', 'parameters', 'seconds']]

def plot_music(names):
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
    for name in names:
        h = pd.read_csv(Path(music_runs[name]['record']['artifacts']) / 'history.csv')
        axes[0].plot(h.epoch, h.train_mse, label=name)
        axes[1].plot(h.epoch, h.val_mse, label=name)
    for ax, title in zip(axes, ['Train', 'Validation']):
        ax.set(xlabel='Эпоха', ylabel='MSE, годы²', title=title, yscale='log')
        ax.legend()
    plt.tight_layout()
    plt.show()

## 3. Нормализация - 1 балл

Сравните три варианта одной регрессионной MLP: A - исходные признаки и годы,
B - нормирована только цель, C - нормированы цель и признаки. Архитектура,
оптимизатор, learning rate, число эпох, batch size и seed одинаковы. Выход линейный.

Сравнивайте MSE/RMSE в годах, постройте общие графики.

In [ ]:
run_music('A: без нормализации', standardized=False, normalized_target=False)
run_music('B: нормирована цель', standardized=False, normalized_target=True)
run_music('C: нормированы X и y')
plot_music(list(music_runs))
music_table()

**Ответьте:** что изменилось в A → B и B → C? Что видно на кривых?
Почему уменьшение loss в нормированной шкале само по себе не доказывает улучшения качества?
Ограничьте выводы выбранным бюджетом обучения.

**Ваш ответ:**

## 4. Кластеризация треков - 1 балл

На `X_train_std` обучите K-means для 3 и 5 кластеров (`n_init=10`, `random_state=SEED`).
Назначьте кластеры validation-трекам и сравните silhouette (`sample_size=1000`, `random_state=SEED`).
Год выпуска в обучение кластеризации не включайте.

Для выбранного варианта составьте validation-профили: размер, доля, медиана года,
25-й и 75-й процентили. Постройте boxplot годов по кластерам. Сохраните сравнение,
профили и выбранные параметры в `results_homework/clustering`.

**Допишите два блока.**

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

CLUSTER_DIR = RESULTS_DIR / 'clustering'
CLUSTER_DIR.mkdir(exist_ok=True)
# >>> ВАШ КОД: cluster_models, cluster_comparison, chosen_k и cluster_labels для validation.
# <<< ВАШ КОД

In [ ]:
# >>> ВАШ КОД: cluster_profiles, boxplot и сохранение результатов.
# <<< ВАШ КОД

**Ответьте:** какое число кластеров выбрали и почему? Насколько перекрываются годы выпуска
в разных группах? Достаточно ли этих данных, чтобы назвать группы жанрами или эпохами?

**Ваш ответ:**

### Бонусные эксперименты с MLP - до 2 баллов суммарно

Каждый пункт - до 1 балла. Можно выбрать не более двух. Выполняйте до итогового test части A.
В каждом сравнении меняйте один фактор и сохраняйте параметры и результаты.

1. **Оптимизатор.** Сравните Adam и SGD с momentum, выделив по два learning rate и одинаковое число эпох.
2. **Ширина и Dropout.** Сравните C с сетью вдвое шире, затем добавьте к широкой сети Dropout.
   Разберите train/validation-кривые и стоимость усложнения модели.
3. **Ограничение выхода.** На нормированных X и y сравните линейный выход и sigmoid.
   Обсудите диапазон возможных предсказаний.

In [ ]:
# Ваш дополнительный эксперимент (необязательно).

## 5. Выбор модели и test для музыки - 0,5 балла

Объедините `music_baselines.set_index('name')` и `music_table()`, выберите метод по validation.
До следующей ячейки укажите `chosen_music` и объясните выбор. После выбора один раз оцените
метод на test вместе с константой и Ridge. Дальнейший подбор по test не проводится.

**Допишите выбор и ответьте, какие изменения помогли больше всего. Высокий score не требуется.**

In [ ]:
# >>> ВАШ КОД: итоговая validation-таблица и chosen_music.
chosen_music = None  # 'Константа', 'Ridge' или имя запуска в music_runs.
# <<< ВАШ КОД

**Обоснование выбора и выводы:**

In [ ]:
X_test, y_test = sample['X_test'], sample['y_test']
X_test_std = feature_scaler.transform(X_test).astype(np.float32)
music_test_predictions = {'Константа': np.full(len(y_test), constant_year),
                          'Ridge': ridge.predict(X_test_std)}
if chosen_music not in music_test_predictions:
    run = music_runs[chosen_music]
    features = X_test_std if run['config']['standardized'] else X_test
    music_test_predictions[chosen_music] = music_predictions(
        run['model'].to(DEVICE), features, run['decode'], run['config']['batch_size'])
music_test = pd.DataFrame([
    dict(name=name, mse=mean_squared_error(y_test, pred),
         rmse=np.sqrt(mean_squared_error(y_test, pred)))
    for name, pred in music_test_predictions.items()
]).set_index('name')
music_test.to_csv(RESULTS_DIR / 'music_test.csv')
music_test

# Часть B. Cora и графовые модели - 5 баллов

Задача: определить тему научной статьи. У вершины есть текстовые признаки, рёбра задают цитирования.
Сравните три источника информации: признаки вершин (MLP), структура связей (Node2Vec),
признаки вместе со связями (GCN).

В разбиении Planetoid/public 140 train, 500 validation и 1000 test-узлов; метки остальных вершин
в этом эксперименте не используются. Все 2708 вершин, их признаки и связи доступны при обучении.
Loss использует только train-метки. Это transductive постановка.

In [ ]:
import pickle
import scipy.sparse as sp
import networkx as nx
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

CORA_DIR = Path('data_cora')
CORA_DIR.mkdir(exist_ok=True)
CORA_EPOCHS = 200
CORA_LR = 0.01
CORA_HIDDEN = 32
CORA_DROPOUT = 0.5
CORA_WEIGHT_DECAY = 5e-4

def load_cora(data_dir):
    cache = data_dir / 'cora_public.npz'
    if cache.exists():
        with np.load(cache) as data:
            return {name: data[name] for name in data.files}
    base_url = 'https://raw.githubusercontent.com/kimiyoung/planetoid/master/data/'
    parts = {}
    for name in ['x', 'y', 'tx', 'ty', 'allx', 'ally', 'graph']:
        path = data_dir / f'ind.cora.{name}'
        if not path.exists():
            urlretrieve(base_url + path.name, path)
        with path.open('rb') as source:
            parts[name] = pickle.load(source, encoding='latin1')
    path = data_dir / 'ind.cora.test.index'
    if not path.exists():
        urlretrieve(base_url + path.name, path)
    order = np.loadtxt(path, dtype=int)
    sorted_order = np.sort(order)
    features = sp.vstack([parts['allx'], parts['tx']]).tolil()
    features[order] = features[sorted_order]
    targets = np.vstack([parts['ally'], parts['ty']])
    targets[order] = targets[sorted_order]
    row, col = zip(*[(i, j) for i, neighbors in parts['graph'].items() for j in neighbors])
    data = dict(X=features.toarray().astype(np.float32), y=targets.argmax(axis=1),
                row=np.asarray(row), col=np.asarray(col),
                train=np.arange(len(parts['y'])),
                val=np.arange(len(parts['y']), len(parts['y']) + 500), test=sorted_order)
    np.savez_compressed(cache, **data)
    return data

cora = load_cora(CORA_DIR)
cora_X = cora['X'] / np.maximum(cora['X'].sum(axis=1, keepdims=True), 1)
cora_y = cora['y']
cora_train_idx, cora_val_idx, cora_test_idx = cora['train'], cora['val'], cora['test']
cora_features = torch.as_tensor(cora_X, dtype=torch.float32, device=DEVICE)
cora_targets = torch.as_tensor(cora_y, dtype=torch.long, device=DEVICE)
cora_train_mask = torch.zeros(len(cora_y), dtype=torch.bool, device=DEVICE)
cora_val_mask = torch.zeros(len(cora_y), dtype=torch.bool, device=DEVICE)
cora_train_mask[cora_train_idx] = True
cora_val_mask[cora_val_idx] = True
print('Cora:', cora_X.shape, '| train/val/test:',
      len(cora_train_idx), len(cora_val_idx), len(cora_test_idx))

## 6. Структура Cora - 1 балл

**Сборка матрицы, 0,5 балла.** По `row`, `col` и числу вершин соберите разреженную
неориентированную матрицу смежности без петель. Повторяющиеся связи должны давать вес 1.
Строка i признаков соответствует вершине i графа. Верните SciPy CSR-матрицу.

**Анализ, 0,5 балла.** Выведите число вершин, рёбер, компонент связности, размер крупнейшей
компоненты и число изолированных вершин. Постройте распределение степеней.

**Допишите функцию и ячейку анализа.**

In [ ]:
def build_cora_graph(row, col, n_nodes):
    # >>> ВАШ КОД: бинарная симметричная CSR-матрица без петель.
    pass
    # <<< ВАШ КОД

cora_A = build_cora_graph(cora['row'], cora['col'], len(cora_y))
cora_graph = nx.from_scipy_sparse_array(cora_A)

In [ ]:
# >>> ВАШ КОД: статистики Cora и распределение степеней.
# <<< ВАШ КОД

**Ответьте:** что структура графа говорит о доступности соседей для разных вершин?
Как обрабатывать изолированную вершину при агрегации?

**Ваш ответ:**

### Готовые функции оценки Cora

Классификация оценивается по accuracy и macro-F1. Состояние нейросети выбирается по validation
macro-F1; train/test-метрики не используются для выбора эпохи. Цикл вызывает уже реализованные
в части A `train_step` и `save_progress`.

In [ ]:
def cora_scores(prediction, indices):
    return dict(accuracy=accuracy_score(cora_y[indices], prediction[indices]),
                macro_f1=f1_score(cora_y[indices], prediction[indices],
                                  labels=np.arange(7), average='macro', zero_division=0))

def predict_cora(name):
    run = cora_runs[name]
    if 'classifier' in run:
        return run['classifier'].predict(run['embedding'])
    run['model'].eval()
    with torch.no_grad():
        return run['model'](cora_features).argmax(dim=1).cpu().numpy()

def cora_table():
    table = pd.DataFrame([run['record'] for run in cora_runs.values()]).set_index('name')
    return table.reindex(columns=['val_accuracy', 'val_macro_f1', 'best_epoch', 'parameters', 'seconds'])

In [ ]:
def plot_cora(names):
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
    for name in names:
        h = pd.read_csv(Path(cora_runs[name]['record']['artifacts']) / 'history.csv')
        axes[0].plot(h.epoch, h.train_loss, label=name)
        axes[1].plot(h.epoch, h.val_macro_f1, label=name)
    axes[0].set(xlabel='Эпоха', ylabel='Train loss')
    axes[1].set(xlabel='Эпоха', ylabel='Validation macro-F1')
    for ax in axes:
        ax.legend()
    plt.tight_layout()
    plt.show()

## 7. MLP без рёбер и обучающая маска - 0,5 балла

MLP собирается вашей `make_mlp`. Новая задача здесь - правильно ограничить loss обучающими
вершинами. Все logits имеют форму `[2708, 7]`; в cross-entropy должны попадать только строки
с `cora_train_mask=True` и соответствующие им метки. Допишите `cora_loss`.

После функций запуска ниже обучите базовую модель и сохраните validation accuracy/macro-F1.

In [ ]:
def cora_loss(logits, target):
    # >>> ВАШ КОД: cross-entropy только для train-узлов.
    pass
    # <<< ВАШ КОД

#### Общий запуск нейросетей на Cora

In [ ]:
def run_cora(name, factory, config, epochs=None, lr=None, weight_decay=None):
    epochs = CORA_EPOCHS if epochs is None else epochs
    lr = CORA_LR if lr is None else lr
    weight_decay = CORA_WEIGHT_DECAY if weight_decay is None else weight_decay
    seed_everything(SEED)
    model = factory().to(DEVICE)
    optimizer = make_optimizer(model, 'Adam', lr, weight_decay)
    config = dict(config, part='cora', name=name, seed=SEED, epochs=epochs,
                  lr=lr, weight_decay=weight_decay, split='Planetoid/public')
    folder = make_run_folder(config)
    history, best_state, best_score, best_epoch = [], None, -np.inf, None
    started = time.perf_counter()
    for epoch in range(1, epochs + 1):
        model.train()
        loss, _ = train_step(model, optimizer, cora_loss, cora_features, cora_targets, DEVICE)
        model.eval()
        with torch.no_grad():
            logits = model(cora_features)
            val_loss = nn.functional.cross_entropy(logits[cora_val_mask], cora_targets[cora_val_mask]).item()
            prediction = logits.argmax(dim=1).cpu().numpy()
        train_metrics = cora_scores(prediction, cora_train_idx)
        val_metrics = cora_scores(prediction, cora_val_idx)
        history.append(dict(epoch=epoch, train_loss=loss.item(), val_loss=val_loss,
                            train_accuracy=train_metrics['accuracy'], train_macro_f1=train_metrics['macro_f1'],
                            val_accuracy=val_metrics['accuracy'], val_macro_f1=val_metrics['macro_f1']))
        is_best = val_metrics['macro_f1'] > best_score
        if is_best:
            best_score, best_epoch = val_metrics['macro_f1'], epoch
            best_state = copy.deepcopy(model.state_dict())
        save_progress(history, model, folder, is_best)
    model.load_state_dict(best_state)
    best = pd.DataFrame(history).loc[best_epoch - 1]
    record = dict(config, val_accuracy=best.val_accuracy, val_macro_f1=best.val_macro_f1,
                  best_epoch=best_epoch, parameters=sum(p.numel() for p in model.parameters()),
                  seconds=time.perf_counter() - started, artifacts=str(folder))
    cora_runs[name] = dict(model=model, record=record)
    pd.Series(record).to_json(folder / 'result.json', force_ascii=False, indent=2)
    print(name, '| validation macro-F1:', round(best_score, 3))
    return record

In [ ]:
run_cora('MLP/features',
         lambda: make_mlp(in_features=cora_X.shape[1], out_features=7,
                          hidden=CORA_HIDDEN, dropout=CORA_DROPOUT),
         dict(kind='mlp', layers=2, hidden=CORA_HIDDEN, dropout=CORA_DROPOUT))
cora_table()

**Ответьте:** какие узлы участвуют в loss, а какие только в выборе модели?
Какую информацию о графе использует эта базовая модель?

**Ваш ответ:**

## 8. Node2Vec и классификатор - 1 балл

Получите embeddings вершин по полному графу без использования меток.
Используйте `Node2Vec`: `dimensions=64`, `walk_length=20`,
`num_walks=10`, `p=1`, `q=0.5`, `workers=1`, `seed=SEED`. При `fit` задайте `window=5`,
`min_count=1`, `epochs=5`, `sg=1`, `seed=SEED`.

Соберите массив `node_embeddings` в порядке вершин `0, ..., 2707`. Обучите
`LogisticRegression(C=1, max_iter=1000)` на train-узлах, оцените validation.
Сохраните embeddings, параметры и итоговые метрики. Посредственные значения score допустимы.

**Допишите создание embeddings и обучение `node_classifier`.**

In [ ]:
from node2vec import Node2Vec

# >>> ВАШ КОД: node_embeddings и node_classifier.
# <<< ВАШ КОД

In [ ]:
node_folder = make_run_folder(dict(part='cora', name='Node2Vec/structure', dimensions=64,
                                   walk_length=20, num_walks=10, p=1, q=0.5, workers=1,
                                   window=5, min_count=1, epochs=5, sg=1, classifier_C=1,
                                   classifier_max_iter=1000, seed=SEED))
node_prediction = node_classifier.predict(node_embeddings)
node_val = cora_scores(node_prediction, cora_val_idx)
node_record = dict(name='Node2Vec/structure', kind='node2vec', val_accuracy=node_val['accuracy'],
                   val_macro_f1=node_val['macro_f1'], artifacts=str(node_folder))
cora_runs['Node2Vec/structure'] = dict(classifier=node_classifier, embedding=node_embeddings, record=node_record)
np.save(node_folder / 'embeddings.npy', node_embeddings)
pd.Series(node_record).to_json(node_folder / 'result.json', force_ascii=False, indent=2)
cora_table()

**Ответьте:** какую информацию использует Node2Vec, которой нет у MLP?
Как `p` и `q` влияют на обход графа? Какие ограничения есть у такого представления
для изолированных и новых вершин?

**Ваш ответ:**

## 9. GCN: нормировка и агрегация - 1,5 балла

**Нормировка, 0,5 балла.** Добавьте self-loops и вычислите

$$
\widetilde A=A+I,\qquad
\widetilde D_{ii}=\sum_j\widetilde A_{ij},\qquad
\widehat A=\widetilde D^{-1/2}\widetilde A\widetilde D^{-1/2}.
$$

`normalize_graph` должна вернуть `torch.sparse_coo_tensor` с типом float32.

**Агрегация, 1 балл.** Допишите `GCN.forward`: перед каждым линейным слоем агрегируйте
представления через `torch.sparse.mm(self.adjacency, features)`. Между слоями применяйте
ReLU и заданный Dropout. Последний слой возвращает 7 логитов без активации.
Конструктор и цикл обучения уже даны.

Матрица \(\widehat A\) фиксирована, обучаются веса линейных слоёв. Выбор эпохи,
маска loss и сохранение результатов те же, что у MLP на Cora.

**Допишите нормировку графа.**

In [ ]:
def normalize_graph(adjacency):
    # >>> ВАШ КОД: self-loops, симметричная нормировка и разреженный torch-тензор.
    pass
    # <<< ВАШ КОД

**Допишите агрегацию в `forward`.**

In [ ]:
class GCN(nn.Module):
    def __init__(self, adjacency, in_features, hidden=32, out_features=7, layers=2, dropout=0.5):
        super().__init__()
        self.register_buffer('adjacency', adjacency)
        dims = [in_features] + [hidden] * (layers - 1) + [out_features]
        self.layers = nn.ModuleList([nn.Linear(a, b) for a, b in zip(dims[:-1], dims[1:])])
        self.dropout = nn.Dropout(dropout)

    def forward(self, features):
        # >>> ВАШ КОД: агрегация, Linear, ReLU/Dropout между слоями.
        pass
        # <<< ВАШ КОД

In [ ]:
cora_normalized = normalize_graph(cora_A)
run_cora('GCN/original/2',
         lambda: GCN(cora_normalized, cora_X.shape[1], hidden=CORA_HIDDEN,
                     layers=2, dropout=CORA_DROPOUT),
         dict(kind='gcn', graph='original', layers=2, hidden=CORA_HIDDEN, dropout=CORA_DROPOUT))
cora_table()

**Ответьте:** какую информацию добавляет один слой агрегации? Откуда приходит информация
за два слоя? Какова роль self-loops и нормировки? Сопоставьте GCN с MLP и Node2Vec по validation.

**Ваш ответ:**

## 10. Роль рёбер, глубина и итоговая оценка - 1 балл

Проверьте роль структуры, сохранив признаки, метки, маски, seed и настройки обучения.
Готовый код ниже строит два контроля: граф без связей между разными вершинами
и граф с перемешанными рёбрами при сохранении степеней каждой вершины.
Self-loops в обоих случаях добавляет ваша `normalize_graph`.

1. Обучите двухслойные GCN на обоих контрольных графах и сравните с исходной GCN и MLP.
2. На исходном графе обучите GCN с одним и четырьмя слоями; сравните с двумя слоями.
3. Сведите validation-метрики всех методов в одну таблицу. Для MLP и GCN также укажите число
   параметров и постройте кривые обучения.
   Веса каждого запуска должны обучаться с нуля.

In [ ]:
rewired_graph = cora_graph.copy()
nx.double_edge_swap(rewired_graph, nswap=5 * cora_graph.number_of_edges(),
                    max_tries=50 * cora_graph.number_of_edges(), seed=SEED)
rewired_A = nx.to_scipy_sparse_array(rewired_graph, nodelist=range(len(cora_y)),
                                    dtype=np.float32, format='csr')
cora_controls = {
    'self_only': normalize_graph(sp.csr_matrix(cora_A.shape, dtype=np.float32)),
    'rewired': normalize_graph(rewired_A),
}

**Допишите запуски двух контрольных графов и двух дополнительных глубин.**

In [ ]:
# >>> ВАШ КОД: GCN/self_only/2, GCN/rewired/2, GCN/original/1, GCN/original/4.
# Используйте run_cora с теми же hidden, dropout, epochs, lr и weight_decay.
# <<< ВАШ КОД

In [ ]:
plot_cora(['MLP/features', 'GCN/original/2', 'GCN/self_only/2', 'GCN/rewired/2',
           'GCN/original/1', 'GCN/original/4'])
cora_table()

**Ответьте:** что изменилось после удаления и перемешивания рёбер? Зачем сохранять степени
при перемешивании? Как результат GCN с одними self-loops связан с MLP той же архитектуры?
Что наблюдается при увеличении глубины и чем можно объяснить изменение train/validation-кривых?

**Ваш ответ:**

### Итоговый test Cora

Выберите метод по validation, запишите `chosen_cora` и объясните выбор **до** следующей ячейки.
Затем один раз оцените его на test. Не выбирайте лучший метод по test-таблице.
Балл за раздел не зависит от того, какая модель победила.

In [ ]:
# >>> ВАШ КОД: имя метода из cora_runs, выбранного по validation.
chosen_cora = None
# <<< ВАШ КОД

**Обоснование выбора:**

In [ ]:
cora_test_prediction = predict_cora(chosen_cora)
cora_test = pd.DataFrame([dict(name=chosen_cora, **cora_scores(cora_test_prediction, cora_test_idx))])
cora_test.to_csv(RESULTS_DIR / 'cora_test.csv', index=False)
cora_test

## Бонус. Анекдот - 0,1 балла

Расскажите хороший анекдот. Если проверяющему будет смешно, получите 0,1 балла.
Нейрослоп - БАН!

Ваш анекдот:

## Описание применения генеративной модели

Если использовали ИИ при выполнении ДЗ, заполните пункты ниже. Если не использовали, напишите: «ИИ при выполнении работы не использовался».

1. **Инструменты.** Укажите названия моделей или сервисов, версии (если известны) и ссылки.
2. **Части работы.** Для каждого инструмента перечислите функции, ячейки или ответы, в которых использовали ИИ. Уточните, были ли они подготовлены с его помощью целиком или частично.
3. **Задачи и способ применения.** Что поручали ИИ и как использовали результат: объяснение, генерация кода или текста, редактирование, поиск ошибок?
4. **Ваш вклад.** Что сделали самостоятельно, что взяли из ответа ИИ и что изменили?
5. **Проверка.** Как проверяли код, вычисления и объяснения? Какие ошибки ИИ нашли и исправили?
6. **Польза и ограничения.** Чем ИИ помог, что стало понятнее и где его помощь оказалась бесполезной?

**Ваш ответ:**

## Источники и материалы
Может чем-то поможет, может что-то почитаете

- Основа части [A](https://github.com/xiyori/intro-to-dl-hse/blob/2024-2025/homeworks-small/shw-01-mlp-light.ipynb).
- Основа части [B](https://github.com/esokolov/ml-course-hse/blob/master/ml2-2026-spring/seminars/sem04-graph.ipynb).
- Данные Year Prediction MSD: Bertin-Mahieux, T. (2011), [страница датасета](https://archive.ics.uci.edu/dataset/203/yearpredictionmsd), [DOI](https://doi.org/10.24432/C50K61), CC BY 4.0.
- Данные Cora и public split: [репозиторий Planetoid](https://github.com/kimiyoung/planetoid/tree/master/data).